# EIP-7904 Project Attribution — Who Is Affected and What Can They Do

The previous notebooks quantified *how many* transactions diverge under EIP-7904 repricing and *what kinds* of divergences occur. This notebook answers the harder question: **which real-world projects are affected, and what can they do about it?**

We map every divergence back to a named project, rank projects by impact severity, classify the remediation pathway each project faces, and identify the specific caller-callee pairs that need outreach. The goal is to produce an **actionable triage list** — not just statistics, but a roadmap for who needs to hear about EIP-7904 and what they should do.

**Related notebooks:**
- `01_impact_overview.ipynb` — headline numbers and cross-cutting statistics
- `02_status_changes.ipynb` — deep dive on success→fail transitions
- `03_call_tree_analysis.ipynb` — call tree divergence patterns

In [ ]:
from helpers import *
import numpy as np
from plotly.subplots import make_subplots

# Load key tables
outreach = read_table("outreach_priority.csv")
project_owner = read_table("project_owner_summary.csv")
failure_pairs = read_table("status_failure_call_pairs_labeled.csv")
intermediaries = read_table("intermediary_breakpoints.csv")

# Sample the large Sankey edges file
sankey_edges = pd.read_csv(TABLES_DIR / "project_sankey_edges.csv", nrows=5000)

# Headline numbers from DuckDB
total_divergences = query_scalar("SELECT count(*) FROM hot_7904")
total_status_changed = query_scalar("SELECT count(*) FROM hot_7904 WHERE status_changed = true")

print(f"Outreach priority list:   {len(outreach):,} projects")
print(f"Project-owner combos:     {len(project_owner):,} rows")
print(f"Failure caller-callee:    {len(failure_pairs):,} pairs")
print(f"Intermediary breakpoints: {len(intermediaries):,} addresses")
print(f"Sankey edges (sampled):   {len(sankey_edges):,} rows")
print(f"\nTotal divergences:        {fmt_count(total_divergences)}")
print(f"Status-changed (broken):  {fmt_count(total_status_changed)}")

## 1. Outreach Priority Ranking

Not all affected projects are equal. We compute a **priority score** that blends status-changed transaction count, total divergent transactions, and aggregate gas delta into a single ranking. Projects at the top of this list are the ones that need to hear about EIP-7904 first — either because their users will experience broken transactions, or because the scale of behavioral change is large enough to warrant review.

The scoring methodology weights status-changed (broken) transactions most heavily, since these represent real user-facing failures, with divergent transaction volume and gas impact as secondary signals.

In [ ]:
# Top 20 projects by priority score
top_outreach = outreach.nlargest(20, "priority_score").sort_values("priority_score", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top_outreach["project"],
    x=top_outreach["priority_score"],
    orientation="h",
    marker_color=COLORS["broken"],
    opacity=0.85,
    text=[f"{s:,.0f}" for s in top_outreach["priority_score"]],
    textposition="outside",
    customdata=np.stack([
        top_outreach["status_changed_txs"].values,
        top_outreach["divergent_txs"].values,
        [fmt_gas(g) for g in top_outreach["total_gas_delta"].values],
    ], axis=-1),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Priority score: %{x:,.0f}<br>"
        "Status-changed txs: %{customdata[0]:,}<br>"
        "Divergent txs: %{customdata[1]:,}<br>"
        "Total gas delta: %{customdata[2]}<br>"
        "<extra></extra>"
    ),
))
fig.update_layout(**plotly_layout(
    title="Top 20 Projects by Outreach Priority Score",
    xaxis_title="Priority Score",
    height=600,
    width=950,
))
fig.show()

# Print the top 5 with detail
print("Top 5 priority projects:\n")
for _, row in outreach.nlargest(5, "priority_score").iterrows():
    print(f"  {row['project']}")
    print(f"    Priority score:     {row['priority_score']:,.0f}")
    print(f"    Status-changed txs: {fmt_count(row['status_changed_txs'])}")
    print(f"    Divergent txs:      {fmt_count(row['divergent_txs'])}")
    print(f"    Total gas delta:    {fmt_gas(row['total_gas_delta'])}")
    print()

## 2. Status Changes by Project

Status-changed transactions are the sharpest signal of impact — these are transactions that **succeed today but would fail** under EIP-7904. The bar chart below shows the top 20 projects by status-changed count, colored by their **owner bucket** — who is responsible for fixing the issue.

Owner buckets classify the remediation *actor*: is it the project itself that needs to update its contracts? A proxy admin who can push an upgrade? A front-door router that controls gas forwarding? Or an upstream integrator whose gas budget is too tight?

In [ ]:
# Top 20 projects by status-changed txs, colored by owner bucket
status_by_project = (
    project_owner
    .groupby(["divergence_project", "owner_bucket"])["status_changed_txs"]
    .sum()
    .reset_index()
)

# Get top 20 projects by total status_changed
top_status_projects = (
    status_by_project
    .groupby("divergence_project")["status_changed_txs"]
    .sum()
    .nlargest(20)
    .index
)
plot_df = status_by_project[status_by_project["divergence_project"].isin(top_status_projects)]

# Pivot for stacked bar
pivot = (
    plot_df
    .pivot_table(index="divergence_project", columns="owner_bucket",
                 values="status_changed_txs", fill_value=0)
)
# Sort by total
pivot["_total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("_total", ascending=True)
pivot = pivot.drop(columns="_total")

# Assign colors from a palette
bucket_colors = {}
palette = ["#e74c3c", "#3498db", "#27ae60", "#e67e22", "#8e44ad", "#1abc9c", "#f39c12", "#95a5a6"]
for i, bucket in enumerate(pivot.columns):
    bucket_colors[bucket] = palette[i % len(palette)]

fig = go.Figure()
for bucket in pivot.columns:
    fig.add_trace(go.Bar(
        y=pivot.index,
        x=pivot[bucket],
        name=bucket,
        orientation="h",
        marker_color=bucket_colors[bucket],
        opacity=0.85,
    ))

fig.update_layout(**plotly_layout(
    title="Top 20 Projects by Status-Changed (Broken) Transactions",
    xaxis_title="Status-Changed Transactions",
    barmode="stack",
    height=600,
    width=1000,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
))
fig.show()

total_sc = project_owner["status_changed_txs"].sum()
top1 = status_by_project.groupby("divergence_project")["status_changed_txs"].sum().nlargest(1)
print(f"Total status-changed txs across all projects: {fmt_count(total_sc)}")
print(f"Top project ({top1.index[0]}) accounts for {fmt_count(top1.values[0])} ({top1.values[0]/total_sc*100:.1f}%)")

## 3. Divergent Transactions by Project

Not every divergence is a breakage. Many transactions change behavior — different gas consumption, different call tree shapes, different event logs — without flipping from success to failure. The ranking by *total divergent transactions* reveals a different picture than the status-changed ranking: some projects have enormous divergence volumes but very few actual failures.

This distinction matters for prioritization. A project with 100K divergent txs but zero broken ones still needs to know about EIP-7904 (their gas costs are changing), but the urgency is lower than a project with 500 broken transactions.

In [ ]:
# Top 20 projects by divergent txs
div_by_project = (
    project_owner
    .groupby("divergence_project")
    .agg(divergent_txs=("divergent_txs", "sum"), status_changed_txs=("status_changed_txs", "sum"))
    .reset_index()
    .nlargest(20, "divergent_txs")
    .sort_values("divergent_txs", ascending=True)
)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Top 20 by Divergent Transactions",
    "Divergent vs Status-Changed (log scale)",
], horizontal_spacing=0.15)

fig.add_trace(go.Bar(
    y=div_by_project["divergence_project"],
    x=div_by_project["divergent_txs"],
    orientation="h",
    marker_color=COLORS["changed"],
    opacity=0.85,
    text=[fmt_count(v) for v in div_by_project["divergent_txs"]],
    textposition="outside",
    showlegend=False,
), row=1, col=1)

# Scatter: divergent vs status-changed for all projects
all_projects = (
    project_owner
    .groupby("divergence_project")
    .agg(divergent_txs=("divergent_txs", "sum"), status_changed_txs=("status_changed_txs", "sum"))
    .reset_index()
)
fig.add_trace(go.Scatter(
    x=all_projects["divergent_txs"],
    y=all_projects["status_changed_txs"],
    mode="markers",
    marker=dict(color=COLORS["changed"], size=6, opacity=0.6),
    text=all_projects["divergence_project"],
    hovertemplate="<b>%{text}</b><br>Divergent: %{x:,}<br>Status-changed: %{y:,}<extra></extra>",
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Divergent Transactions", row=1, col=1)
fig.update_xaxes(title_text="Divergent Txs", type="log", row=1, col=2)
fig.update_yaxes(title_text="Status-Changed Txs", type="log", row=1, col=2)

fig.update_layout(**plotly_layout(
    height=600, width=1100,
))
fig.show()

# How many projects have divergences but no breakage?
no_break = all_projects[all_projects["status_changed_txs"] == 0]
has_break = all_projects[all_projects["status_changed_txs"] > 0]
print(f"Projects with divergences but NO breakage: {len(no_break)} "
      f"({len(no_break)/len(all_projects)*100:.0f}%)")
print(f"Projects with at least one broken tx:      {len(has_break)} "
      f"({len(has_break)/len(all_projects)*100:.0f}%)")

## 4. Gas Impact by Project

Raw transaction counts do not capture severity. A project might have few divergent transactions but enormous aggregate gas deltas — meaning each affected transaction shifts substantially. The gas delta view highlights projects where the repricing has the deepest per-transaction impact, which correlates with how close those transactions run to their gas limits and how heavily they use the repriced opcodes.

In [ ]:
# Top 20 projects by total gas delta
gas_by_project = outreach.nlargest(20, "total_gas_delta").sort_values("total_gas_delta", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=gas_by_project["project"],
    x=gas_by_project["total_gas_delta"],
    orientation="h",
    marker_color=COLORS["increased"],
    opacity=0.85,
    text=[fmt_gas(g) for g in gas_by_project["total_gas_delta"]],
    textposition="outside",
))
fig.update_layout(**plotly_layout(
    title="Top 20 Projects by Total Gas Delta",
    xaxis_title="Total Gas Delta",
    height=600,
    width=950,
))
fig.show()

# Summary stats
total_gas = outreach["total_gas_delta"].sum()
top5_gas = outreach.nlargest(5, "total_gas_delta")["total_gas_delta"].sum()
print(f"Total gas delta across all projects: {fmt_gas(total_gas)}")
print(f"Top 5 projects account for: {fmt_gas(top5_gas)} ({top5_gas/total_gas*100:.1f}%)")

## 5. Owner Bucket Analysis

Every affected project-divergence combination is assigned an **owner bucket** — the entity best positioned to remediate the issue. Understanding the distribution of owner buckets tells us *who needs to act* at an ecosystem level:

| Bucket | Meaning |
|--------|---------|
| `direct_project_fix` | The project's own contracts need updating |
| `proxy_wallet_or_upgrade_admin` | A proxy admin can push an upgrade to the affected contract |
| `front_door_or_router_fix` | A router/aggregator that forwards calls needs to adjust gas stipends |
| `upstream_integrator_gas_budget` | An upstream project integrating with this one uses too-tight gas budgets |
| `eoa_wallet_gas_estimate` | End-user wallets need to update gas estimation logic |
| `no_action_needed` | The divergence is benign or self-correcting |

In [ ]:
# Owner bucket distribution
owner_agg = (
    project_owner
    .groupby("owner_bucket")
    .agg(
        projects=("divergence_project", "nunique"),
        status_changed_txs=("status_changed_txs", "sum"),
        divergent_txs=("divergent_txs", "sum"),
    )
    .reset_index()
    .sort_values("divergent_txs", ascending=False)
)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    subplot_titles=["Owner Bucket by Divergent Txs", "Owner Bucket by Status-Changed Txs"],
)

# Donut chart — divergent txs
bucket_palette = ["#e74c3c", "#3498db", "#27ae60", "#e67e22", "#8e44ad", "#1abc9c", "#f39c12", "#95a5a6"]
fig.add_trace(go.Pie(
    labels=owner_agg["owner_bucket"],
    values=owner_agg["divergent_txs"],
    hole=0.45,
    marker_colors=bucket_palette[:len(owner_agg)],
    textinfo="label+percent",
    textposition="outside",
    showlegend=False,
), row=1, col=1)

# Horizontal bar — status-changed txs
owner_sc = owner_agg.sort_values("status_changed_txs", ascending=True)
fig.add_trace(go.Bar(
    y=owner_sc["owner_bucket"],
    x=owner_sc["status_changed_txs"],
    orientation="h",
    marker_color=COLORS["broken"],
    opacity=0.85,
    text=[fmt_count(v) for v in owner_sc["status_changed_txs"]],
    textposition="outside",
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Status-Changed Transactions", row=1, col=2)
fig.update_layout(**plotly_layout(
    title="Who Needs to Act? Owner Bucket Distribution",
    height=500, width=1100,
))
fig.show()

# Print summary table
print("Owner bucket summary:\n")
print(f"  {'Bucket':<40} {'Projects':>8} {'Status-Changed':>15} {'Divergent':>12}")
print("  " + "-" * 78)
for _, row in owner_agg.iterrows():
    print(f"  {row['owner_bucket']:<40} {row['projects']:>8} "
          f"{fmt_count(row['status_changed_txs']):>15} {fmt_count(row['divergent_txs']):>12}")

## 6. Remediation Bucket Analysis

While owner buckets tell us *who* should act, **remediation buckets** tell us *what kind of fix* is needed. This is the technical dimension of the triage: can the contract be upgraded in place, does it require a migration, or is it a simple off-chain integration update?

The remediation bucket determines the timeline and feasibility of a fix — an `admin_upgrade_possible` contract can be patched in days, while an `immutable_contract_or_migration` may require deploying an entirely new contract and migrating state.

In [ ]:
# Remediation bucket distribution
remediation_agg = (
    project_owner
    .groupby("remediation_bucket")
    .agg(
        projects=("divergence_project", "nunique"),
        status_changed_txs=("status_changed_txs", "sum"),
        divergent_txs=("divergent_txs", "sum"),
    )
    .reset_index()
    .sort_values("divergent_txs", ascending=False)
)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    subplot_titles=["Remediation Type by Divergent Txs", "Remediation Type by Status-Changed Txs"],
)

# Donut
rem_palette = ["#27ae60", "#2ecc71", "#3498db", "#e67e22", "#e74c3c", "#8e44ad", "#f39c12", "#95a5a6"]
fig.add_trace(go.Pie(
    labels=remediation_agg["remediation_bucket"],
    values=remediation_agg["divergent_txs"],
    hole=0.45,
    marker_colors=rem_palette[:len(remediation_agg)],
    textinfo="label+percent",
    textposition="outside",
    showlegend=False,
), row=1, col=1)

# Horizontal bar — status-changed
rem_sc = remediation_agg.sort_values("status_changed_txs", ascending=True)
fig.add_trace(go.Bar(
    y=rem_sc["remediation_bucket"],
    x=rem_sc["status_changed_txs"],
    orientation="h",
    marker_color=COLORS["call_tree"],
    opacity=0.85,
    text=[fmt_count(v) for v in rem_sc["status_changed_txs"]],
    textposition="outside",
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Status-Changed Transactions", row=1, col=2)
fig.update_layout(**plotly_layout(
    title="What Kind of Fix Is Needed? Remediation Bucket Distribution",
    height=500, width=1100,
))
fig.show()

# Print summary
print("Remediation bucket summary:\n")
print(f"  {'Bucket':<40} {'Projects':>8} {'Status-Changed':>15} {'Divergent':>12}")
print("  " + "-" * 78)
for _, row in remediation_agg.iterrows():
    print(f"  {row['remediation_bucket']:<40} {row['projects']:>8} "
          f"{fmt_count(row['status_changed_txs']):>15} {fmt_count(row['divergent_txs']):>12}")

## 7. Project Interaction Sankey

Divergences do not happen in isolation. A transaction typically flows through multiple projects — a user calls project A, which calls project B, which calls project C. When the repricing causes a divergence deep in the call stack, the *source project* and the *target project* may be different entities with different remediation responsibilities.

The Sankey below visualizes these inter-project flows. Each link represents divergent transactions flowing from a source project to a target project, sized by transaction count. Thick links reveal the dominant cross-project dependency patterns that EIP-7904 stresses.

In [ ]:
# Project interaction Sankey from sampled edges
# Filter out self-loops for clarity
sankey_cross = sankey_edges[sankey_edges["source_project"] != sankey_edges["target_project"]].copy()

fig = plot_sankey(
    sankey_cross,
    columns=["source_project", "target_project"],
    title="Divergence Flow Between Projects (sampled, cross-project only)",
    value_col="divergent_txs",
    min_flow=10,
    top_n=15,
    width=1000,
    height=600,
)
fig.show()

# Summary stats
n_source = sankey_cross["source_project"].nunique()
n_target = sankey_cross["target_project"].nunique()
total_cross = sankey_cross["divergent_txs"].sum()
print(f"Cross-project divergence flows: {len(sankey_cross):,} edges")
print(f"Source projects: {n_source}, Target projects: {n_target}")
print(f"Total divergent txs in cross-project flows: {fmt_count(total_cross)}")

## 8. Failing Caller-Callee Pairs by Project

This is the most actionable data in the notebook. Each row identifies a specific **caller project → callee project** pair where status failures occur — transactions that succeed today but would fail under EIP-7904 because the caller forwards insufficient gas to the callee.

These pairs are the direct outreach targets. For each pair, we know the average gas provided and average gas used, which tells us how tight the margin is and how much the gas stipend needs to increase.

In [ ]:
# Top 20 failing caller→callee pairs
top_pairs = failure_pairs.nlargest(20, "status_failures").copy()
top_pairs["pair_label"] = top_pairs["caller_project"] + " -> " + top_pairs["callee_project"]
top_pairs = top_pairs.sort_values("status_failures", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top_pairs["pair_label"],
    x=top_pairs["status_failures"],
    orientation="h",
    marker_color=COLORS["status"],
    opacity=0.85,
    text=[fmt_count(v) for v in top_pairs["status_failures"]],
    textposition="outside",
    customdata=np.stack([
        [fmt_gas(g) for g in top_pairs["avg_gas_provided"]],
        [fmt_gas(g) for g in top_pairs["avg_gas_used"]],
    ], axis=-1),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Status failures: %{x:,}<br>"
        "Avg gas provided: %{customdata[0]}<br>"
        "Avg gas used: %{customdata[1]}<br>"
        "<extra></extra>"
    ),
))
fig.update_layout(**plotly_layout(
    title="Top 20 Failing Caller -> Callee Project Pairs",
    xaxis_title="Status Failures (Broken Transactions)",
    height=650,
    width=1000,
))
fig.update_yaxes(tickfont=dict(size=10))
fig.show()

# Print the top 10 with gas details
print("Top 10 failing pairs with gas analysis:\n")
for _, row in failure_pairs.nlargest(10, "status_failures").iterrows():
    margin = row["avg_gas_provided"] - row["avg_gas_used"]
    margin_pct = margin / row["avg_gas_provided"] * 100 if row["avg_gas_provided"] > 0 else 0
    print(f"  {row['caller_project']} -> {row['callee_project']}")
    print(f"    Failures: {fmt_count(row['status_failures'])}  |  "
          f"Avg provided: {fmt_gas(row['avg_gas_provided'])}  |  "
          f"Avg used: {fmt_gas(row['avg_gas_used'])}  |  "
          f"Margin: {fmt_gas(margin)} ({margin_pct:.1f}%)")
    print()

## 9. Intermediary Breakpoint Analysis

Some contracts act as **intermediaries** — they sit between the transaction originator and the contract where the divergence manifests. These breakpoint contracts are critical because they often control the gas stipend forwarded to downstream calls. A single intermediary can amplify EIP-7904's impact across many downstream projects.

The `breakpoint_score` captures how influential each intermediary is, weighing the number of breakpoint transactions, the breadth of affected root and downstream projects, and the count of success-flip events.

In [ ]:
# Top intermediary breakpoints
top_intermediaries = intermediaries.nlargest(20, "breakpoint_score").sort_values(
    "breakpoint_score", ascending=True
)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Top 20 Intermediaries by Breakpoint Score",
    "Breakpoint Txs vs Downstream Reach",
], horizontal_spacing=0.15)

# Bar chart
fig.add_trace(go.Bar(
    y=top_intermediaries["project"],
    x=top_intermediaries["breakpoint_score"],
    orientation="h",
    marker_color=COLORS["call_tree"],
    opacity=0.85,
    text=[f"{s:,.0f}" for s in top_intermediaries["breakpoint_score"]],
    textposition="outside",
    showlegend=False,
), row=1, col=1)

# Scatter: breakpoint txs vs downstream reach
fig.add_trace(go.Scatter(
    x=intermediaries["breakpoint_txs"],
    y=intermediaries["distinct_downstream_projects"],
    mode="markers",
    marker=dict(
        color=intermediaries["success_flip_txs"],
        colorscale="YlOrRd",
        size=8,
        opacity=0.7,
        colorbar=dict(title="Success Flips", x=1.02),
    ),
    text=intermediaries["project"],
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Breakpoint txs: %{x:,}<br>"
        "Downstream projects: %{y}<br>"
        "Success flips: %{marker.color:,}<br>"
        "<extra></extra>"
    ),
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Breakpoint Score", row=1, col=1)
fig.update_xaxes(title_text="Breakpoint Transactions", type="log", row=1, col=2)
fig.update_yaxes(title_text="Distinct Downstream Projects", row=1, col=2)

fig.update_layout(**plotly_layout(
    height=600, width=1150,
))
fig.show()

# Print top 5
print("Top 5 intermediary breakpoints:\n")
for _, row in intermediaries.nlargest(5, "breakpoint_score").iterrows():
    print(f"  {row['project']}")
    print(f"    Score: {row['breakpoint_score']:,.0f}  |  "
          f"Breakpoint txs: {fmt_count(row['breakpoint_txs'])}  |  "
          f"Root projects: {row['distinct_root_projects']}  |  "
          f"Downstream: {row['distinct_downstream_projects']}  |  "
          f"Success flips: {fmt_count(row['success_flip_txs'])}")
    print()

## 10. Root Cause Motifs

When a non-root contract first diverges in a transaction, the triplet **(root project, caller project, callee project)** forms a **motif** — a recurring pattern of cross-project interaction that EIP-7904 disrupts. Identifying the dominant motifs tells us which specific integration pathways are most fragile.

The `change_reason` column classifies *why* the divergence occurred at that point in the call stack, and the `depth` tells us how far down the call tree the first non-root change appears. Shallow depths suggest tight gas margins at the integration boundary; deeper ones suggest cascading effects through multiple layers of contracts.

In [ ]:
# Load motifs
motifs = read_table("first_changed_nonroot_motifs.csv")

# Top motifs by txs, grouped by change_reason — stacked bar
motif_agg = (
    motifs
    .groupby(["root_project", "caller_project", "callee_project", "change_reason"])
    .agg(txs=("txs", "sum"), success_flip_txs=("success_flip_txs", "sum"))
    .reset_index()
)
motif_agg["motif"] = (
    motif_agg["root_project"] + " -> "
    + motif_agg["caller_project"] + " -> "
    + motif_agg["callee_project"]
)

# Get top 15 motifs by total txs
top_motif_totals = (
    motif_agg.groupby("motif")["txs"].sum().nlargest(15).index
)
plot_motifs = motif_agg[motif_agg["motif"].isin(top_motif_totals)]

# Pivot for stacked bar
motif_pivot = (
    plot_motifs
    .pivot_table(index="motif", columns="change_reason", values="txs", fill_value=0)
)
motif_pivot["_total"] = motif_pivot.sum(axis=1)
motif_pivot = motif_pivot.sort_values("_total", ascending=True)
motif_pivot = motif_pivot.drop(columns="_total")

reason_colors = {
    "status": COLORS["status"],
    "call_tree": COLORS["call_tree"],
    "event_logs": COLORS["event_logs"],
    "gas_used": COLORS["increased"],
}

fig = go.Figure()
for reason in motif_pivot.columns:
    fig.add_trace(go.Bar(
        y=motif_pivot.index,
        x=motif_pivot[reason],
        name=reason,
        orientation="h",
        marker_color=reason_colors.get(reason, COLORS["neutral"]),
        opacity=0.85,
    ))

fig.update_layout(**plotly_layout(
    title="Top 15 Root Cause Motifs (Root -> Caller -> Callee)",
    xaxis_title="Transactions",
    barmode="stack",
    height=600,
    width=1050,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
))
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

# Depth distribution
depth_dist = motifs.groupby("depth")["txs"].sum().reset_index()
print("Depth distribution of first non-root divergence:\n")
for _, row in depth_dist.sort_values("depth").iterrows():
    print(f"  Depth {int(row['depth'])}: {fmt_count(row['txs'])} txs")

# Change reason summary
reason_summary = motifs.groupby("change_reason").agg(
    txs=("txs", "sum"), success_flips=("success_flip_txs", "sum")
).sort_values("txs", ascending=False)
print(f"\nChange reason summary:")
for reason, row in reason_summary.iterrows():
    print(f"  {reason}: {fmt_count(row['txs'])} txs, {fmt_count(row['success_flips'])} success flips")

## 11. Concentration Analysis

How concentrated is the impact? If the top 10 projects account for 90% of all breakage, then outreach to a small number of teams can address the vast majority of the problem. If impact is spread across hundreds of projects, the remediation effort is fundamentally different.

The cumulative distribution below shows what fraction of total status-changed transactions is captured as we move down the priority-ranked project list.

In [ ]:
# Concentration: cumulative % of status-changed txs
sc_by_project = (
    project_owner
    .groupby("divergence_project")["status_changed_txs"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
sc_by_project["cumulative"] = sc_by_project["status_changed_txs"].cumsum()
total_sc = sc_by_project["status_changed_txs"].sum()
sc_by_project["cumulative_pct"] = sc_by_project["cumulative"] / total_sc * 100
sc_by_project["rank"] = range(1, len(sc_by_project) + 1)

# Only plot projects that contribute to status-changed
sc_nonzero = sc_by_project[sc_by_project["status_changed_txs"] > 0]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sc_nonzero["rank"],
    y=sc_nonzero["cumulative_pct"],
    mode="lines+markers",
    line=dict(color=COLORS["broken"], width=2.5),
    marker=dict(size=5),
    text=sc_nonzero["divergence_project"],
    hovertemplate=(
        "<b>Rank %{x}: %{text}</b><br>"
        "Cumulative: %{y:.1f}%<br>"
        "<extra></extra>"
    ),
))

# Add reference lines
for threshold, label in [(50, "50%"), (80, "80%"), (95, "95%")]:
    rank_at = sc_nonzero.loc[sc_nonzero["cumulative_pct"] >= threshold, "rank"].iloc[0]
    fig.add_hline(y=threshold, line_dash="dot", line_color=COLORS["neutral"], opacity=0.5)
    fig.add_vline(x=rank_at, line_dash="dot", line_color=COLORS["neutral"], opacity=0.5)
    fig.add_annotation(
        x=rank_at, y=threshold,
        text=f"Top {rank_at} = {label}",
        showarrow=True, arrowhead=2,
        ax=40, ay=-25,
        font=dict(size=11),
    )

fig.update_layout(**plotly_layout(
    title="Concentration of Status-Changed Transactions Across Projects",
    xaxis_title="Project Rank (by status-changed txs)",
    yaxis_title="Cumulative % of Status-Changed Txs",
    height=450,
    width=900,
))
fig.show()

# Print key thresholds
print("Concentration thresholds:\n")
for pct in [50, 80, 90, 95, 99]:
    rank = sc_nonzero.loc[sc_nonzero["cumulative_pct"] >= pct, "rank"].iloc[0]
    project_name = sc_nonzero.loc[sc_nonzero["rank"] == rank, "divergence_project"].iloc[0]
    print(f"  Top {rank:>3} projects cover {pct}% of status-changed txs (last: {project_name})")

print(f"\nTotal projects with status-changed txs: {len(sc_nonzero)}")
print(f"Total projects in dataset: {len(sc_by_project)}")

## 12. Owner-Remediation Cross-Analysis

The final analytical view combines the two classification dimensions — **who acts** (owner bucket) and **what they do** (remediation bucket) — into a single heatmap. This reveals the structural patterns in the remediation landscape: for instance, whether most `direct_project_fix` cases require immutable contract migrations or whether admin upgrades suffice.

In [ ]:
# Cross-tab: owner_bucket x remediation_bucket
cross = (
    project_owner
    .groupby(["owner_bucket", "remediation_bucket"])["status_changed_txs"]
    .sum()
    .reset_index()
    .pivot_table(index="owner_bucket", columns="remediation_bucket",
                 values="status_changed_txs", fill_value=0)
)

fig = go.Figure(go.Heatmap(
    z=cross.values,
    x=cross.columns.tolist(),
    y=cross.index.tolist(),
    colorscale="YlOrRd",
    text=cross.values,
    texttemplate="%{text:,}",
    showscale=True,
    colorbar=dict(title="Status-Changed Txs"),
))
fig.update_layout(**plotly_layout(
    title="Owner Bucket x Remediation Bucket (Status-Changed Txs)",
    height=450,
    width=900,
))
fig.update_xaxes(tickangle=30, tickfont=dict(size=10))
fig.update_yaxes(tickfont=dict(size=10))
fig.show()

# Sankey: owner -> remediation flow
owner_rem_flow = (
    project_owner
    .groupby(["owner_bucket", "remediation_bucket"])["status_changed_txs"]
    .sum()
    .reset_index()
)
owner_rem_flow.columns = ["source", "target", "value"]
# Prefix to avoid name collisions
owner_rem_flow["source"] = "Owner: " + owner_rem_flow["source"]
owner_rem_flow["target"] = "Rem: " + owner_rem_flow["target"]

fig2 = plot_sankey(
    owner_rem_flow,
    columns=["source", "target"],
    title="Remediation Flow: Owner Bucket -> Remediation Bucket (Status-Changed Txs)",
    value_col="value",
    min_flow=5,
    width=950,
    height=500,
)
fig2.show()

## Key Findings and Outreach Recommendations

### Impact Concentration
- EIP-7904's breakage is **highly concentrated**: a small number of projects account for the vast majority of status-changed transactions. Targeted outreach to the top-ranked projects can address most of the problem.
- Many projects have large divergence counts but zero breakage — they need to be *informed* but not urgently remediated.

### Owner Landscape
- The owner bucket distribution reveals whether the ecosystem can self-heal. Projects where the fix lies with a proxy admin or an upstream integrator have a clear remediation path; projects requiring immutable contract migration face a harder road.
- Intermediary contracts that forward gas to downstream projects are a critical leverage point — fixing a single high-traffic intermediary can resolve breakage across many downstream projects simultaneously.

### Actionable Outreach List
1. **Top priority-score projects**: Begin outreach with the highest-ranked projects in the outreach priority table. These combine high breakage counts, large divergence volumes, and significant gas deltas.
2. **Failing caller-callee pairs**: The specific caller-to-callee pairs identified in section 8 are the most actionable data points. Each pair identifies exactly which project needs to increase its gas stipend when calling which downstream project.
3. **Intermediary breakpoints**: High-score intermediaries (section 9) represent single points of failure that affect many downstream projects. Fixing these yields outsized returns.

### Remediation Pathways
- **Admin-upgradeable contracts**: Can be patched relatively quickly once the project team is aware. The outreach message should include the specific gas delta and affected function signatures.
- **Immutable contracts**: Require migration to a new deployment. The outreach message should emphasize the timeline pressure — these fixes take longer and need to start sooner.
- **Wallet/integration updates**: The easiest category — off-chain gas estimation updates. Wallet developers and bundler operators should be notified en masse.

### Root Cause Patterns
- The dominant motifs (section 10) reveal that a handful of cross-project integration patterns account for most of the breakage. These are the structural dependencies that EIP-7904 stresses, and they should inform both the outreach strategy and any design adjustments to the EIP itself.